# Phase 2 rerun: PeerQA with all PDFs, and gemma with the model-card document prompt

In [ ]:
# Setup: clone the repo and copy the datasets (same layout as retrieval-benchmark-full).
# The Hugging Face token comes from Kaggle Secrets (Add-ons > Secrets > HF_TOKEN); never paste it here.
!pip install -q "datasets<4.0.0" qdrant_client
!git clone https://github.com/Scrowbin/science-document-viewer
!cd science-document-viewer && git checkout feat/app-improvements && git log --oneline -1
!pip install -q -r science-document-viewer/backend/requirements.txt
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["DEBUG"] = "True"  # backend settings need DEBUG or a SECRET_KEY
!cp -r /kaggle/input/datasets/kristiko/qasper-official /kaggle/working/science-document-viewer/eval/data/
!mv /kaggle/working/science-document-viewer/eval/data/qasper-official /kaggle/working/science-document-viewer/eval/data/qasper_official
!mkdir -p /kaggle/working/science-document-viewer/eval/pdfs
!cp -r /kaggle/input/datasets/kristiko/peerqa /kaggle/working/science-document-viewer/eval/pdfs/peerqa
!find /kaggle/working/science-document-viewer/eval/pdfs/peerqa -name "*.pdf" | wc -l   # expect about 107

PeerQA: the first full run found only 8/107 papers. The script now searches `eval/pdfs/peerqa` recursively and prints how many PDFs it found (should be close to 107).

In [ ]:
!python science-document-viewer/eval/phases/phase_2_retrieval/scripts/evaluate_retrieval.py --cohort PEERQA-PDF

Gemma on C-RET again: the document prompt is now `title: none | text: ` (EmbeddingGemma model card). Written to `c_ret_gemma_doc_prompt/` and `summary_gemma_doc_prompt.json`, so the full C-RET run is not overwritten.

In [ ]:
!python science-document-viewer/eval/phases/phase_2_retrieval/scripts/evaluate_retrieval.py --cohort C-RET --models gemma --run-tag gemma_doc_prompt

In [ ]:
# Zip the results and download them from the Output tab, then unzip into the repo at the same path.
!cd science-document-viewer && zip -qr /kaggle/working/phase2_artifacts.zip eval/phases/phase_2_retrieval/artifacts